# 09 — Dual-Stream Face + Mouth V2 (FAST TRAINING)

โมเดล V2 สำหรับ Thai Visual Speech Recognition

```text
Full Face 112×112 ── Face CNN ──┐
                                ├─ Feature Fusion → BiGRU → CTC → Raw Thai Text
Mouth ROI 96×96 ── Mouth CNN ───┘
```

แนวคิด:
- ใช้ **ทั้งใบหน้า** เพื่อเก็บข้อมูลจากแก้ม กราม สีหน้า และการเคลื่อนไหวรอบปาก
- ใช้ **Mouth ROI** เป็นสาขาหลักเพื่อรักษารายละเอียดริมฝีปาก
- เปลี่ยน BatchNorm → **GroupNorm** เพื่อเสถียรกว่ากับ batch เล็ก
- ใช้ AMP (`float16`) บน CUDA เพื่อให้เทรนเร็ว
- filter CTC-invalid clips อัตโนมัติ
- เก็บ raw VSR text ไว้ให้ LLM แก้ภาษาในขั้นถัดไป

> Notebook นี้ไม่เขียนทับโมเดล V1

In [ ]:
# Cell 1 — Environment
import sys, os, json, math, random, re, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("=" * 80)
print("ENVIRONMENT")
print("=" * 80)
print("Python :", sys.executable)
print("Version:", sys.version.split()[0])
print("Torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Notebook นี้ออกแบบให้ใช้ CUDA")

DEVICE = torch.device("cuda")
print("GPU    :", torch.cuda.get_device_name(0))
print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

In [ ]:
# Cell 2 — Path / Config
PROJECT_ROOT = Path(r"D:\Face_detection")
VIDEO_ID = "video001"

SPLIT_ROOT = PROJECT_ROOT / "dataset" / "splits" / VIDEO_ID
TRAIN_CSV = SPLIT_ROOT / "train.csv"
VAL_CSV   = SPLIT_ROOT / "val.csv"
TEST_CSV  = SPLIT_ROOT / "test.csv"

RAW_VIDEO = PROJECT_ROOT / "dataset" / "raw_videos" / f"{VIDEO_ID}.webm"

# ใช้ detection ที่ผ่าน speaker mapping แล้วถ้ามี
DETECTION_CANDIDATES = [
    PROJECT_ROOT / "dataset" / "reid" / VIDEO_ID / "final_2speaker" / "final_2speaker_detection.csv",
    PROJECT_ROOT / "dataset" / "reid" / VIDEO_ID / "reidentified_detection.csv",
    PROJECT_ROOT / "dataset" / "yolo_face_detection" / VIDEO_ID / "detection.csv",
]
DETECTION_CSV = next((p for p in DETECTION_CANDIDATES if p.exists()), None)

FACE_CACHE_ROOT = PROJECT_ROOT / "dataset" / "face_crops_v2" / VIDEO_ID
MODEL_ROOT = PROJECT_ROOT / "models" / "lipreading_dualstream_v2"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
FACE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

# ---------- SPEED / MEMORY ----------
FACE_SIZE = 112
MOUTH_SIZE = 96

BATCH_SIZE = 2
NUM_WORKERS = 4      # Notebook + Windows: เสถียรที่สุด
MAX_FRAMES = 140      # ลดจาก 180 เพื่อเร็วขึ้น แต่ยังเหลือ timestep มากพอ
GRAD_ACCUM_STEPS = 2

FEATURE_DIM = 96
GRU_HIDDEN = 160
GRU_LAYERS = 2
DROPOUT = 0.15

LR = 3e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 30
PATIENCE = 10
GRAD_CLIP = 5.0
SEED = 42

# face crop ขยายกรอบเล็กน้อยเพื่อเก็บกราม/แก้ม
FACE_MARGIN = 0.12

for p in [TRAIN_CSV, VAL_CSV, TEST_CSV, RAW_VIDEO]:
    print(p, "->", p.exists())

print("DETECTION_CSV :", DETECTION_CSV)
print("FACE_CACHE    :", FACE_CACHE_ROOT)
print("MODEL_ROOT    :", MODEL_ROOT)

if DETECTION_CSV is None:
    raise FileNotFoundError(
        "ไม่พบ detection CSV กรุณาตรวจ dataset/reid/video001 หรือ yolo_face_detection/video001"
    )

In [ ]:
# Cell 3 — Seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("Seed ready")

In [ ]:
# Cell 4 — Load Split CSV
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))
print("\nColumns:")
print(train_df.columns.tolist())

required = {"clip_id", "speaker_id", "text", "frame_paths"}
missing = required - set(train_df.columns)
if missing:
    raise KeyError(f"train.csv ขาด columns: {sorted(missing)}")

display(train_df.head(3))

In [ ]:
# Cell 5 — Character Tokenizer
BLANK_TOKEN = "<blank>"
UNK_TOKEN = "<unk>"

train_texts = train_df["text"].fillna("").astype(str).tolist()
charset = sorted(set("".join(train_texts)))
charset = [ch for ch in charset if ch not in ["\n", "\r", "\t"]]

itos = [BLANK_TOKEN, UNK_TOKEN] + charset
stoi = {token: idx for idx, token in enumerate(itos)}

BLANK_ID = stoi[BLANK_TOKEN]
UNK_ID = stoi[UNK_TOKEN]
VOCAB_SIZE = len(itos)

def encode_text(text):
    return [stoi.get(ch, UNK_ID) for ch in str(text)]

def decode_target(ids):
    out = []
    for idx in ids:
        idx = int(idx)
        if idx == BLANK_ID:
            continue
        tok = itos[idx]
        out.append("�" if tok == UNK_TOKEN else tok)
    return "".join(out)

def ctc_greedy_decode(logits, lengths):
    pred_ids = logits.argmax(dim=-1)
    results = []
    for b in range(pred_ids.size(0)):
        seq = pred_ids[b, :int(lengths[b])].tolist()
        collapsed, prev = [], None
        for idx in seq:
            if idx != prev and idx != BLANK_ID:
                collapsed.append(idx)
            prev = idx
        results.append(decode_target(collapsed))
    return results

with open(MODEL_ROOT / "charset.json", "w", encoding="utf-8") as f:
    json.dump(
        {"itos": itos, "blank_id": BLANK_ID, "unk_id": UNK_ID},
        f, ensure_ascii=False, indent=2
    )

print("VOCAB_SIZE:", VOCAB_SIZE)
print("BLANK_ID  :", BLANK_ID)

In [ ]:
# Cell 6 — TRUE CTC compatibility + filter
# CTC ต้องมี timestep >= target_length + จำนวน adjacent repeated labels

def min_ctc_steps(text):
    ids = encode_text(text)
    repeats = sum(ids[i] == ids[i-1] for i in range(1, len(ids)))
    return len(ids) + repeats

def available_steps(row):
    paths = [p for p in str(row["frame_paths"]).split("|") if p.strip()]
    return min(len(paths), MAX_FRAMES)

def filter_ctc_valid(df, name):
    keep = []
    bad = []
    for idx, row in df.iterrows():
        steps = available_steps(row)
        need = min_ctc_steps(str(row["text"]))
        if steps >= need:
            keep.append(idx)
        else:
            bad.append((row["clip_id"], steps, need, row["text"]))
    out = df.loc[keep].reset_index(drop=True)
    print(f"{name}: {len(df)} -> {len(out)} | removed={len(bad)}")
    if bad:
        print("ตัวอย่างที่ถูกตัด:")
        for item in bad[:10]:
            print(item)
    return out

train_df = filter_ctc_valid(train_df, "Train")
val_df   = filter_ctc_valid(val_df, "Val")
test_df  = filter_ctc_valid(test_df, "Test")

## Cell 7 — สร้าง Face Cache ครั้งเดียว

ส่วนนี้อ่าน raw video เพียงครั้งเดียว แล้ว crop ใบหน้าตาม detection CSV เพื่อให้ตอน Train ไม่ต้องเปิดวิดีโอซ้ำทุก epoch

ไฟล์จะเก็บที่:

`dataset/face_crops_v2/video001/speaker_XXX/frame_XXXXXXXX.jpg`

ถ้ามี cache แล้ว Cell นี้จะข้ามไฟล์เดิม ทำให้รอบถัดไปเร็วมาก

In [ ]:
# Cell 7 — Build Face Cache
det_df = pd.read_csv(DETECTION_CSV)

needed_cols = {"frame_index", "x1", "y1", "x2", "y2"}
if not needed_cols.issubset(det_df.columns):
    raise KeyError(f"Detection CSV ต้องมี {sorted(needed_cols)}")

# สร้าง mapping frame -> detections
# ถ้ามี speaker_id จะใช้ speaker_id ให้ตรงกับ split
has_speaker = "speaker_id" in det_df.columns

# หา frame indices ที่ถูกใช้จริงจาก split ทั้งหมด
all_split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
needed_frames_by_speaker = {}

frame_re = re.compile(r"frame_(\d+)", re.IGNORECASE)

for _, row in all_split_df.iterrows():
    spk = int(row["speaker_id"])
    for p in str(row["frame_paths"]).split("|"):
        m = frame_re.search(Path(p).stem)
        if m:
            fi = int(m.group(1))
            needed_frames_by_speaker.setdefault(spk, set()).add(fi)

total_needed = sum(len(v) for v in needed_frames_by_speaker.values())
print("Needed face frames:", total_needed)
print("Speakers:", sorted(needed_frames_by_speaker))

# Detection lookup
if has_speaker:
    det_df = det_df[det_df["speaker_id"].notna()].copy()
    det_df["speaker_id"] = det_df["speaker_id"].astype(int)

lookup = {}
for _, r in det_df.iterrows():
    fi = int(r["frame_index"])
    spk = int(r["speaker_id"]) if has_speaker else None
    key = (fi, spk)
    # ถ้ามีหลาย detection เลือก confidence สูงสุด
    score = float(r["det_score"]) if "det_score" in r and pd.notna(r["det_score"]) else 0.0
    old = lookup.get(key)
    if old is None or score > old["score"]:
        lookup[key] = {
            "bbox": (float(r["x1"]), float(r["y1"]), float(r["x2"]), float(r["y2"])),
            "score": score
        }

def face_cache_path(speaker_id, frame_index):
    d = FACE_CACHE_ROOT / f"speaker_{int(speaker_id):03d}"
    d.mkdir(parents=True, exist_ok=True)
    return d / f"frame_{int(frame_index):08d}.jpg"

# ตรวจว่ามี cache ครบหรือยัง
missing_requests = []
for spk, frames in needed_frames_by_speaker.items():
    for fi in frames:
        if not face_cache_path(spk, fi).exists():
            missing_requests.append((spk, fi))

print("Already cached:", total_needed - len(missing_requests))
print("Need to build  :", len(missing_requests))

if missing_requests:
    wanted_by_frame = {}
    for spk, fi in missing_requests:
        wanted_by_frame.setdefault(fi, []).append(spk)

    cap = cv2.VideoCapture(str(RAW_VIDEO))
    if not cap.isOpened():
        raise RuntimeError(f"เปิดวิดีโอไม่ได้: {RAW_VIDEO}")

    frame_idx = 0
    saved = 0
    missing_det = 0
    max_needed = max(wanted_by_frame)

    while frame_idx <= max_needed:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx in wanted_by_frame:
            h, w = frame.shape[:2]

            for spk in wanted_by_frame[frame_idx]:
                rec = lookup.get((frame_idx, spk))
                if rec is None and not has_speaker:
                    # fallback: ใช้ detection ที่ confidence สูงสุดในเฟรมนั้น
                    candidates = [
                        v for (fi, s), v in lookup.items()
                        if fi == frame_idx
                    ]
                    rec = max(candidates, key=lambda x: x["score"]) if candidates else None

                if rec is None:
                    missing_det += 1
                    continue

                x1, y1, x2, y2 = rec["bbox"]
                fw, fh = x2 - x1, y2 - y1
                x1 -= fw * FACE_MARGIN
                x2 += fw * FACE_MARGIN
                y1 -= fh * FACE_MARGIN
                y2 += fh * FACE_MARGIN

                x1 = max(0, int(round(x1)))
                y1 = max(0, int(round(y1)))
                x2 = min(w, int(round(x2)))
                y2 = min(h, int(round(y2)))

                if x2 <= x1 or y2 <= y1:
                    continue

                crop = frame[y1:y2, x1:x2]
                crop = cv2.resize(crop, (FACE_SIZE, FACE_SIZE), interpolation=cv2.INTER_AREA)
                crop = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

                cv2.imwrite(str(face_cache_path(spk, frame_idx)), crop)
                saved += 1

        frame_idx += 1
        if frame_idx % 1000 == 0:
            print(f"Read frame {frame_idx:,} | saved={saved:,}")

    cap.release()
    print("Saved face crops:", saved)
    print("Missing detection:", missing_det)
else:
    print("Face cache complete — skip preprocessing ✅")

In [ ]:
import re

frame_re = re.compile(
    r"frame_(\d+)",
    re.IGNORECASE
)

In [ ]:
# Cell 8 — Dataset: Face + Mouth
class DualStreamDataset(Dataset):
    def __init__(self, dataframe, project_root, max_frames=140, train=False):
        self.df = dataframe.reset_index(drop=True).copy()
        self.project_root = Path(project_root)
        self.max_frames = max_frames
        self.train = train

    def __len__(self):
        return len(self.df)

    def resolve(self, p):
        p = Path(str(p))
        return p if p.is_absolute() else self.project_root / p

    def select_paths(self, paths):
        if len(paths) <= self.max_frames:
            return paths
        idx = np.linspace(0, len(paths)-1, self.max_frames).astype(int)
        return [paths[i] for i in idx]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spk = int(row["speaker_id"])

        mouth_paths = [p for p in str(row["frame_paths"]).split("|") if p.strip()]
        mouth_paths = self.select_paths(mouth_paths)

        face_frames = []
        mouth_frames = []

        for p in mouth_paths:
            mp = self.resolve(p)
            m = frame_re.search(mp.stem)
            if m is None:
                continue
            fi = int(m.group(1))
            fp = face_cache_path(spk, fi)

            mouth = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
            face = cv2.imread(str(fp), cv2.IMREAD_GRAYSCALE)

            if mouth is None or face is None:
                continue

            if mouth.shape != (MOUTH_SIZE, MOUTH_SIZE):
                mouth = cv2.resize(mouth, (MOUTH_SIZE, MOUTH_SIZE), interpolation=cv2.INTER_AREA)
            if face.shape != (FACE_SIZE, FACE_SIZE):
                face = cv2.resize(face, (FACE_SIZE, FACE_SIZE), interpolation=cv2.INTER_AREA)

            # lightweight augmentation เฉพาะ train
            if self.train and random.random() < 0.35:
                alpha = random.uniform(0.90, 1.10)
                beta = random.uniform(-8, 8)
                mouth = cv2.convertScaleAbs(mouth, alpha=alpha, beta=beta)
                face = cv2.convertScaleAbs(face, alpha=alpha, beta=beta)

            mouth_frames.append(mouth)
            face_frames.append(face)

        if not mouth_frames:
            raise RuntimeError(f"No paired face/mouth frames: {row['clip_id']}")

        face = torch.from_numpy(np.stack(face_frames).astype(np.float32) / 255.0).unsqueeze(1)
        mouth = torch.from_numpy(np.stack(mouth_frames).astype(np.float32) / 255.0).unsqueeze(1)

        text = str(row["text"])
        target = torch.tensor(encode_text(text), dtype=torch.long)

        return {
            "face": face,
            "mouth": mouth,
            "target": target,
            "text": text,
            "clip_id": row["clip_id"],
            "speaker_id": spk,
        }

def collate_fn(batch):
    lengths = torch.tensor([x["face"].shape[0] for x in batch], dtype=torch.long)
    target_lengths = torch.tensor([x["target"].numel() for x in batch], dtype=torch.long)
    max_t = int(lengths.max())
    B = len(batch)

    face = torch.zeros(B, max_t, 1, FACE_SIZE, FACE_SIZE, dtype=torch.float32)
    mouth = torch.zeros(B, max_t, 1, MOUTH_SIZE, MOUTH_SIZE, dtype=torch.float32)

    for i, x in enumerate(batch):
        t = x["face"].shape[0]
        face[i, :t] = x["face"]
        mouth[i, :t] = x["mouth"]

    return {
        "face": face,
        "mouth": mouth,
        "video_lengths": lengths,
        "targets": torch.cat([x["target"] for x in batch]),
        "target_lengths": target_lengths,
        "texts": [x["text"] for x in batch],
        "clip_ids": [x["clip_id"] for x in batch],
    }

print("Dataset ready")

In [ ]:
print("DualStreamDataset:", DualStreamDataset)
print("Train samples:", len(train_df))
print("NUM_WORKERS:", NUM_WORKERS)
print("MAX_FRAMES:", MAX_FRAMES)

In [ ]:
# Cell 9 — DataLoaders (SAFE FOR WINDOWS/JUPYTER)

NUM_WORKERS = 0

train_dataset = DualStreamDataset(
    train_df,
    PROJECT_ROOT,
    MAX_FRAMES,
    train=True
)

val_dataset = DualStreamDataset(
    val_df,
    PROJECT_ROOT,
    MAX_FRAMES,
    train=False
)

test_dataset = DualStreamDataset(
    test_df,
    PROJECT_ROOT,
    MAX_FRAMES,
    train=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn
)

print("✅ DataLoaders created")
print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))
print("Test batches :", len(test_loader))

## Cell 10 — Dual-Stream Model

ใช้ GroupNorm แทน BatchNorm เพื่อหลีกเลี่ยงปัญหา batch เล็ก และใช้ encoder ขนาดเล็กเพื่อให้ RTX 4050 6GB เทรนได้เร็ว

In [ ]:
# Cell 10 — Model
class SmallVisualEncoder(nn.Module):
    def __init__(self, in_ch=1, out_dim=96):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 24, 5, padding=2),
            nn.GroupNorm(6, 24),
            nn.SiLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(24, 48, 3, padding=1),
            nn.GroupNorm(8, 48),
            nn.SiLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(48, 96, 3, padding=1),
            nn.GroupNorm(12, 96),
            nn.SiLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.proj = nn.Linear(96, out_dim)

    def forward(self, x):
        x = self.net(x).flatten(1)
        return self.proj(x)

class DualStreamLipReadingV2(nn.Module):
    def __init__(
        self, vocab_size,
        feature_dim=96,
        hidden_dim=160,
        gru_layers=2,
        dropout=0.15
    ):
        super().__init__()

        self.face_encoder = SmallVisualEncoder(1, feature_dim)
        self.mouth_encoder = SmallVisualEncoder(1, feature_dim)

        # learnable gate: ให้โมเดลเรียนเองว่าจะใช้ face/mouth มากแค่ไหน
        self.fusion = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim * 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(feature_dim * 2, feature_dim)
        )

        self.temporal = nn.GRU(
            input_size=feature_dim,
            hidden_size=hidden_dim,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if gru_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, face, mouth, lengths):
        B, T = face.shape[:2]

        f = self.face_encoder(face.reshape(B*T, 1, FACE_SIZE, FACE_SIZE))
        m = self.mouth_encoder(mouth.reshape(B*T, 1, MOUTH_SIZE, MOUTH_SIZE))

        x = torch.cat([f, m], dim=-1)
        x = self.fusion(x).reshape(B, T, -1)

        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.temporal(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=T
        )

        return self.classifier(self.dropout(out))

model = DualStreamLipReadingV2(
    VOCAB_SIZE, FEATURE_DIM, GRU_HIDDEN, GRU_LAYERS, DROPOUT
).to(DEVICE)

params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nParameters: {params:,}")
print("Device:", next(model.parameters()).device)

In [ ]:
# Cell 11 — Loss / Optimizer / Scheduler / AMP
criterion = nn.CTCLoss(
    blank=BLANK_ID,
    reduction="mean",
    zero_infinity=True
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

scaler = torch.amp.GradScaler("cuda")

print("Training objects ready")

In [ ]:
# Cell 12 — CER helpers
def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(
                cur[-1] + 1,
                prev[j] + 1,
                prev[j-1] + (ca != cb)
            ))
        prev = cur
    return prev[-1]

def corpus_cer(refs, preds):
    errors = 0
    chars = 0
    for r, p in zip(refs, preds):
        errors += edit_distance(r, p)
        chars += len(r)
    return errors / max(chars, 1)

print("CER helper ready")

In [ ]:
# Cell 13 — Train / Evaluate

def train_one_epoch(model, loader):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running = 0.0
    count = 0

    for step, batch in enumerate(loader, 1):
        face = batch["face"].to(DEVICE, non_blocking=True)
        mouth = batch["mouth"].to(DEVICE, non_blocking=True)
        lengths = batch["video_lengths"]
        targets = batch["targets"].to(DEVICE, non_blocking=True)
        target_lengths = batch["target_lengths"]

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(face, mouth, lengths)

            log_probs = logits.log_softmax(-1).transpose(0, 1)

            loss = criterion(
                log_probs,
                targets,
                lengths,
                target_lengths
            )

            loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if step % GRAD_ACCUM_STEPS == 0 or step == len(loader):
            scaler.unscale_(optimizer)

            nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

        running += loss.item() * GRAD_ACCUM_STEPS
        count += 1

        # แสดงสถานะทุก 10 steps
        if step % 10 == 0 or step == len(loader):
            print(
                f"\rtrain {step:4d}/{len(loader)} | "
                f"loss={running / max(count, 1):.4f}",
                end="",
                flush=True
            )

    print()

    return running / max(count, 1)


@torch.no_grad()
def evaluate(model, loader, max_decode_batches=None):
    model.eval()

    losses = []
    refs = []
    preds = []

    blank_steps = 0
    total_steps = 0

    for bi, batch in enumerate(loader):
        face = batch["face"].to(
            DEVICE,
            non_blocking=True
        )

        mouth = batch["mouth"].to(
            DEVICE,
            non_blocking=True
        )

        lengths = batch["video_lengths"]

        targets = batch["targets"].to(
            DEVICE,
            non_blocking=True
        )

        target_lengths = batch["target_lengths"]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(
                face,
                mouth,
                lengths
            )

            log_probs = (
                logits
                .log_softmax(-1)
                .transpose(0, 1)
            )

            loss = criterion(
                log_probs,
                targets,
                lengths,
                target_lengths
            )

        losses.append(float(loss.item()))

        pred_ids = logits.argmax(-1).cpu()

        for i, L in enumerate(lengths):
            valid = pred_ids[i, :int(L)]

            blank_steps += int(
                (valid == BLANK_ID).sum()
            )

            total_steps += valid.numel()

        if (
            max_decode_batches is None
            or bi < max_decode_batches
        ):
            batch_preds = ctc_greedy_decode(
                logits.float().cpu(),
                lengths
            )

            preds.extend(batch_preds)
            refs.extend(batch["texts"])

    cer = (
        corpus_cer(refs, preds)
        if refs
        else float("nan")
    )

    blank_ratio = (
        blank_steps
        / max(total_steps, 1)
    )

    return (
        np.mean(losses),
        cer,
        blank_ratio,
        preds,
        refs
    )


print("Train/eval functions ready")

In [ ]:
# Cell 14 — Smoke Test (ไม่ต้องโหลด batch จริง)

print("=" * 60)
print("SMOKE TEST")
print("=" * 60)

model.eval()

# สร้างข้อมูลจำลองแทนการอ่านภาพจาก disk
B = 1
T = 40

face_test = torch.randn(
    B, T, 1, FACE_SIZE, FACE_SIZE,
    device=DEVICE
)

mouth_test = torch.randn(
    B, T, 1, MOUTH_SIZE, MOUTH_SIZE,
    device=DEVICE
)

lengths_test = torch.tensor(
    [T],
    dtype=torch.long
)

print("Face :", face_test.shape)
print("Mouth:", mouth_test.shape)

with torch.no_grad():
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16
    ):
        logits = model(
            face_test,
            mouth_test,
            lengths_test
        )

print("Output:", logits.shape)

assert logits.shape[0] == B
assert logits.shape[1] == T
assert logits.shape[2] == VOCAB_SIZE

print("\n✅ Forward test PASS")
print("✅ Model is ready for training")

## Cell 15 — FULL TRAINING

รัน Cell นี้เมื่อตรวจ Smoke Test ผ่านแล้ว

Checkpoint:
- `best_model.pt`
- `last_model.pt`
- `training_history.csv`
- `model_config.json`

In [ ]:
import re

frame_re = re.compile(
    r"frame_(\d+)",
    re.IGNORECASE
)

print("✅ frame_re ready")

In [ ]:
# ===== FIX: face_cache_path =====

def face_cache_path(speaker_id, frame_index):
    d = FACE_CACHE_ROOT / f"speaker_{int(speaker_id):03d}"
    d.mkdir(parents=True, exist_ok=True)

    return d / f"frame_{int(frame_index):08d}.jpg"


print("✅ face_cache_path ready")

In [ ]:
print(face_cache_path(0, 123))

In [ ]:
print("Testing real dataset...")

sample = train_dataset[0]

print("✅ Dataset sample PASS")
print("Face :", sample["face"].shape)
print("Mouth:", sample["mouth"].shape)
print("Text :", sample["text"])

In [ ]:
batch_test = next(iter(train_loader))

print("✅ DataLoader PASS")
print("Face :", batch_test["face"].shape)
print("Mouth:", batch_test["mouth"].shape)
print("Text :", batch_test["texts"])

In [ ]:
# Cell 15 — Full Training
BEST_PATH = MODEL_ROOT / "best_model.pt"
LAST_PATH = MODEL_ROOT / "last_model.pt"
HISTORY_PATH = MODEL_ROOT / "training_history.csv"

history = []
best_val = float("inf")
bad_epochs = 0

print("=" * 90)
print("START DUAL-STREAM V2 TRAINING")
print("=" * 90)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    train_loss = train_one_epoch(model, train_loader)
    val_loss, val_cer, val_blank, val_preds, val_refs = evaluate(model, val_loader)

    scheduler.step(val_loss)
    lr_now = optimizer.param_groups[0]["lr"]
    mins = (time.time() - t0) / 60

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_cer": val_cer,
        "val_blank_ratio": val_blank,
        "lr": lr_now,
        "minutes": mins
    })
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train={train_loss:.4f} | "
        f"val={val_loss:.4f} | "
        f"CER={val_cer:.4f} | "
        f"blank={val_blank*100:.1f}% | "
        f"lr={lr_now:.2e} | "
        f"{mins:.1f} min"
    )

    for r, p in list(zip(val_refs, val_preds))[:2]:
        print("REF :", r)
        print("PRED:", p)
    print("-" * 90)

    torch.save({
        "model_state_dict": model.state_dict(),
        "epoch": epoch,
        "vocab_size": VOCAB_SIZE,
        "itos": itos,
        "config": {
            "face_size": FACE_SIZE,
            "mouth_size": MOUTH_SIZE,
            "feature_dim": FEATURE_DIM,
            "gru_hidden": GRU_HIDDEN,
            "gru_layers": GRU_LAYERS,
            "dropout": DROPOUT,
        }
    }, LAST_PATH)

    if val_loss < best_val:
        best_val = val_loss
        bad_epochs = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "val_loss": val_loss,
            "val_cer": val_cer,
            "val_blank_ratio": val_blank,
            "vocab_size": VOCAB_SIZE,
            "itos": itos,
            "config": {
                "face_size": FACE_SIZE,
                "mouth_size": MOUTH_SIZE,
                "feature_dim": FEATURE_DIM,
                "gru_hidden": GRU_HIDDEN,
                "gru_layers": GRU_LAYERS,
                "dropout": DROPOUT,
            }
        }, BEST_PATH)
        print("✅ Saved BEST:", BEST_PATH)
    else:
        bad_epochs += 1

    if bad_epochs >= PATIENCE:
        print(f"Early stopping: no improvement for {PATIENCE} epochs")
        break

print("TRAINING FINISHED")

In [ ]:
# ===== DEBUG: ดูว่าค้างตรงไหน =====
import time

print("TEST 1: เริ่มโหลด batch แรก...", flush=True)

t0 = time.time()

batch_debug = next(iter(train_loader))

print(
    f"✅ TEST 1 PASS: โหลด batch แรกสำเร็จ ใช้เวลา {time.time()-t0:.2f} วินาที",
    flush=True
)

print("Face :", batch_debug["face"].shape)
print("Mouth:", batch_debug["mouth"].shape)
print("Text :", batch_debug["texts"][:2])


print("\nTEST 2: ส่งข้อมูลเข้า GPU...", flush=True)

t0 = time.time()

face = batch_debug["face"].to(DEVICE)
mouth = batch_debug["mouth"].to(DEVICE)
lengths = batch_debug["video_lengths"]

print(
    f"✅ TEST 2 PASS: เข้า GPU สำเร็จ ใช้เวลา {time.time()-t0:.2f} วินาที",
    flush=True
)


print("\nTEST 3: Forward...", flush=True)

t0 = time.time()

model.train()

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    logits = model(
        face,
        mouth,
        lengths
    )

print(
    f"✅ TEST 3 PASS: Forward สำเร็จ ใช้เวลา {time.time()-t0:.2f} วินาที",
    flush=True
)

print("Logits:", logits.shape)


print("\nTEST 4: คำนวณ CTC Loss...", flush=True)

targets = batch_debug["targets"].to(DEVICE)
target_lengths = batch_debug["target_lengths"]

t0 = time.time()

with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    log_probs = logits.log_softmax(-1).transpose(0, 1)

    loss = criterion(
        log_probs,
        targets,
        lengths,
        target_lengths
    )

print(
    f"✅ TEST 4 PASS: Loss={loss.item():.4f} ใช้เวลา {time.time()-t0:.2f} วินาที",
    flush=True
)


print("\nTEST 5: Backward...", flush=True)

optimizer.zero_grad(set_to_none=True)

t0 = time.time()

scaler.scale(loss).backward()

print(
    f"✅ TEST 5 PASS: Backward สำเร็จ ใช้เวลา {time.time()-t0:.2f} วินาที",
    flush=True
)

optimizer.zero_grad(set_to_none=True)

print("\n🎉 ทุกขั้นตอนผ่าน")

In [ ]:
# Cell 16 — Test Best Model
ckpt = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])

test_loss, test_cer, test_blank, test_preds, test_refs = evaluate(model, test_loader)

print("=" * 80)
print("FINAL TEST — DUAL STREAM V2")
print("=" * 80)
print(f"Test Loss  : {test_loss:.4f}")
print(f"Test CER   : {test_cer:.4f}")
print(f"Blank Ratio: {test_blank:.4f} ({test_blank*100:.2f}%)")

for i, (r, p) in enumerate(zip(test_refs, test_preds)):
    print(f"\n[{i:03d}]")
    print("REF :", r)
    print("PRED:", p)
    if i >= 9:
        break

In [ ]:
# Cell 17 — Save Model Config
config = {
    "version": "v2-dualstream-face-mouth-fast",
    "architecture": "Dual CNN (Face+Mouth) + BiGRU + CTC",
    "face_size": FACE_SIZE,
    "mouth_size": MOUTH_SIZE,
    "max_frames": MAX_FRAMES,
    "feature_dim": FEATURE_DIM,
    "gru_hidden": GRU_HIDDEN,
    "gru_layers": GRU_LAYERS,
    "dropout": DROPOUT,
    "blank_id": BLANK_ID,
    "vocab_size": VOCAB_SIZE,
    "llm_post_correction": True,
    "tts_after_correction": True,
}

with open(MODEL_ROOT / "model_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved:", MODEL_ROOT / "model_config.json")
print("\nPipeline target:")
print("Face+Mouth → VSR raw text → LLM correction → Display → TTS")

# การปรับให้เร็วขึ้นถ้า VRAM เหลือ

หลังรัน 1 epoch แล้วดู Task Manager / `nvidia-smi`:

- ถ้า VRAM ใช้ไม่เกิน ~4.5 GB → ลอง `BATCH_SIZE = 3`
- ถ้า CUDA OOM → ลด `BATCH_SIZE = 1`
- ถ้า epoch ช้าเกินไป → ลด `MAX_FRAMES = 120`
- **อย่าลด MAX_FRAMES จนต่ำกว่า CTC minimum** เพราะ Cell 6 จะ filter ออก

เป้าหมาย V2 รอบแรก:
1. Blank Ratio ต้องลดลงจาก V1 อย่างชัดเจน
2. CER ต้องต่ำกว่า V1
3. Raw output ต้องมีข้อมูลมากพอให้ LLM แก้คำ ไม่ใช่ output สั้น 1–2 ตัวอักษร